# 07 - FloodNet Segmentation (SegFormer-B2)

Kaggle notebook for training FloodNet flood/road segmentation model.

In [ ]:
import os
import sys
from pathlib import Path

IS_KAGGLE = os.path.exists('/kaggle/input')

if IS_KAGGLE:
    RAW_DIR = Path('/kaggle/input/aerial-imagery-dataset-floodnet-challenge')  # Adjust slug
    OUTPUT_DIR = Path('/kaggle/working')
    sys.path.insert(0, '/kaggle/working')
    !pip install -q segmentation-models-pytorch albumentations transformers
else:
    RAW_DIR = Path('../data/raw/floodnet')
    OUTPUT_DIR = Path('../outputs')
    sys.path.insert(0, str(Path('../src').resolve()))

print(f'Raw dir: {RAW_DIR} (exists: {RAW_DIR.exists()})')
print(f'Output dir: {OUTPUT_DIR}')

In [ ]:
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')

from rescuevision.data.floodnet_preprocess import FloodNetDataset
from rescuevision.data.transforms import get_segmentation_transforms
from rescuevision.models.losses import WeightedCEDiceLoss
from rescuevision.training.train_segmentation import train_segmentation
from rescuevision.utils.seed import set_seed
from rescuevision.constants import NUM_FLOODNET_CLASSES, FLOODNET_CLASSES

from transformers import SegformerForSemanticSegmentation

In [ ]:
config = {
    "seed": 42,
    "device": "cuda",
    "dataset": {
        "image_size": 512,
        "batch_size": 4,
        "num_workers": 2,
        "num_classes": NUM_FLOODNET_CLASSES,
    },
    "model": {
        "backbone": "nvidia/segformer-b2-finetuned-ade-512-512",
        "num_labels": NUM_FLOODNET_CLASSES,
    },
    "training": {
        "epochs": 50,
        "lr": 6e-5,
        "weight_decay": 1e-4,
        "scheduler": "cosine",
        "mixed_precision": True,
        "early_stopping_patience": 8,
        "save_best_metric": "miou",
        "gradient_accumulation_steps": 2,
    },
    "loss": {
        "ce_weight": 0.5,
        "dice_weight": 0.5,
    },
    "augmentation": {
        "horizontal_flip": 0.5,
        "vertical_flip": 0.5,
        "rotate90": 0.5,
        "brightness_contrast": 0.2,
        "normalize": True,
    },
}

set_seed(config["seed"])
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# Create datasets
train_dataset = FloodNetDataset(
    images_dir=RAW_DIR / 'train' / 'images',
    masks_dir=RAW_DIR / 'train' / 'masks',
    transform=get_segmentation_transforms(config['augmentation'], 'train', 512),
    image_size=512,
)

val_dataset = FloodNetDataset(
    images_dir=RAW_DIR / 'val' / 'images',
    masks_dir=RAW_DIR / 'val' / 'masks',
    transform=get_segmentation_transforms(config['augmentation'], 'val', 512),
    image_size=512,
)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train: {len(train_dataset)} samples, Val: {len(val_dataset)} samples')

In [ ]:
# SegFormer model with wrapper
segformer = SegformerForSemanticSegmentation.from_pretrained(
    config['model']['backbone'],
    num_labels=config['model']['num_labels'],
    ignore_mismatched_sizes=True,
)

class SegFormerWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        out = self.model(x)
        logits = out.logits
        logits = F.interpolate(logits, size=x.shape[2:], mode='bilinear', align_corners=False)
        return logits

model = SegFormerWrapper(segformer)

loss_fn = WeightedCEDiceLoss(
    ce_weight=config['loss']['ce_weight'],
    dice_weight=config['loss']['dice_weight'],
    ignore_index=255,
)

output_dir = OUTPUT_DIR / 'floodnet_segformer'

best_metrics = train_segmentation(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    loss_fn=loss_fn,
    config=config,
    output_dir=str(output_dir),
    is_binary=False,
    num_classes=NUM_FLOODNET_CLASSES,
    ignore_index=255,
)

print(f'\nBest metrics: {best_metrics}')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from rescuevision.constants import FLOODNET_COLOR_MAP

model.eval()
model.to(device)

fig, axes = plt.subplots(3, 4, figsize=(20, 15))

for i in range(4):
    batch = next(iter(val_loader))
    images = batch['image'][:1].to(device)
    masks_gt = batch['mask'][:1].numpy()
    
    with torch.no_grad():
        pred = model(images).argmax(1).squeeze().cpu().numpy()
    
    # Colorize
    def colorize(m):
        rgb = np.zeros((*m.shape, 3), dtype=np.uint8)
        for c, color in FLOODNET_COLOR_MAP.items():
            rgb[m == c] = color
        return rgb
    
    img = images[0].cpu().numpy().transpose(1, 2, 0)
    img = (img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406]))
    img = np.clip(img, 0, 1)
    
    axes[0, i].imshow(img)
    axes[0, i].set_title('Image')
    axes[1, i].imshow(colorize(masks_gt[0]))
    axes[1, i].set_title('Ground Truth')
    axes[2, i].imshow(colorize(pred))
    axes[2, i].set_title('Prediction')
    
    for ax in axes[:, i]:
        ax.axis('off')

plt.tight_layout()
plt.savefig(output_dir / 'floodnet_predictions.png', dpi=150, bbox_inches='tight')
plt.show()